# Dataset Statistics

Datapoint counts and label distributions for all benchmark and synthetic datasets.

In [1]:
import os
import re
import pandas as pd
from IPython.display import display

# Resolve project root (two levels up from notebooks/evaluations/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
data_dir = os.path.join(project_root, 'data')
print(f'Project root: {project_root}')

Project root: /mnt/ffs24/home/chidurah/NCEAS_Unsupervised_NLP


## Benchmark Datasets

In [2]:
def load_amazon():
    amz_40 = pd.read_csv(os.path.join(data_dir, 'amazon', 'train_40k.csv'))
    amz_10 = pd.read_csv(os.path.join(data_dir, 'amazon', 'val_10k.csv'))
    amz = pd.concat([amz_40, amz_10])
    amz = amz.drop_duplicates(subset='Title', keep=False).reset_index(drop=True)
    amz = amz.drop_duplicates(subset='productId', keep=False).reset_index(drop=True)
    amz = amz.rename(columns={'Title': 'topic', 'Cat1': 'category_0', 'Cat2': 'category_1', 'Cat3': 'category_2'})
    amz = amz.dropna(subset=['topic', 'category_0', 'category_1', 'category_2']).reset_index(drop=True)
    amz = amz[amz['topic'].apply(lambda x: isinstance(x, str) and x.strip() != '')].reset_index(drop=True)
    return amz


def load_dbpedia():
    def clean(text):
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()
    db = pd.read_csv(os.path.join(data_dir, 'dbpedia', 'DBPEDIA_test.csv'))
    db = db.rename(columns={'text': 'topic', 'l1': 'category_0', 'l2': 'category_1', 'l3': 'category_2'})
    db['topic'] = db['topic'].astype(str).apply(clean)
    return db


def load_arxiv():
    arx = pd.read_csv(os.path.join(data_dir, 'arxiv', 'arxiv_clean.csv'))
    return arx.dropna().reset_index(drop=True)


def load_rcv1():
    rcv1 = pd.read_csv(os.path.join(data_dir, 'rcv1', 'rcv1.csv'))
    rcv1 = rcv1.drop_duplicates(subset='topic', keep=False).reset_index(drop=True)
    rcv1 = rcv1.drop_duplicates(subset='item_id', keep=False).reset_index(drop=True)
    rcv1 = rcv1.dropna().reset_index(drop=True)
    rcv1 = rcv1[rcv1['topic'].apply(lambda x: isinstance(x, str) and x.strip() != '')].reset_index(drop=True)
    return rcv1


def load_wos():
    wos = pd.read_excel(os.path.join(data_dir, 'WebOfScience', 'Data.xlsx'))
    rows = [{'topic': str(r['keywords']), 'category_0': r['Domain'], 'category_1': r['area']} for _, r in wos.iterrows()]
    return pd.DataFrame(rows)


BENCHMARK_CONFIGS = {
    'Amazon':  {'loader': load_amazon,  'depth': 3, 'col_prefix': 'category_'},
    'DBpedia': {'loader': load_dbpedia, 'depth': 3, 'col_prefix': 'category_'},
    'arXiv':   {'loader': load_arxiv,   'depth': 2, 'col_prefix': 'category_'},
    'RCV1':    {'loader': load_rcv1,    'depth': 2, 'col_prefix': 'category_'},
    'WoS':     {'loader': load_wos,     'depth': 2, 'col_prefix': 'category_'},
}

In [3]:
benchmark_rows = []

for name, cfg in BENCHMARK_CONFIGS.items():
    df = cfg['loader']()
    row = {'Dataset': name, 'N': len(df)}
    print(df.columns)
    for i in range(cfg['depth']):
        col = f"{cfg['col_prefix']}{i}"
        row[f'Level {i} labels'] = int(df[col].nunique())
    benchmark_rows.append(row)
    print(f'Loaded {name}: {len(df):,} rows')

benchmark_stats = pd.DataFrame(benchmark_rows).set_index('Dataset')
benchmark_stats

Index(['productId', 'topic', 'userId', 'Helpfulness', 'Score', 'Time', 'Text',
       'category_0', 'category_1', 'category_2'],
      dtype='object')
Loaded Amazon: 14,824 rows
Index(['topic', 'category_0', 'category_1', 'category_2'], dtype='object')
Loaded DBpedia: 60,794 rows
Index(['topic', 'category_0', 'category_1'], dtype='object')
Loaded arXiv: 29,529 rows
Index(['item_id', 'topic', 'category_0', 'category_1'], dtype='object')
Loaded RCV1: 1,566 rows
Index(['topic', 'category_0', 'category_1'], dtype='object')
Loaded WoS: 46,985 rows


,N,Level 0 labels,Level 1 labels,Level 2 labels
Dataset,,,,
Amazon,14824,6,64,454.0
DBpedia,60794,9,70,219.0
arXiv,29529,11,122,NaN
RCV1,1566,58,72,NaN
WoS,46985,7,143,NaN


## Synthetic Datasets

Four datasets derived from two themes × two hierarchy depths:

| Nickname | Theme | Depth | max_sub |
|---|---|---|---|
| ecosystems-shallow | Energy_Ecosystems_and_Humans | 3 | 5 |
| ecosystems-deep | Energy_Ecosystems_and_Humans | 5 | 3 |
| fisheries-shallow | Offshore_energy_impacts_on_fisheries | 3 | 5 |
| fisheries-deep | Offshore_energy_impacts_on_fisheries | 5 | 3 |

In [4]:
synth_base = os.path.join(data_dir, 'synthetic', 'generated_data')

def synth_path(theme, depth, max_sub, t=1.0, synonyms=0, branching='random', noise=0.0):
    fname = f'{theme}_hierarchy_t{t}_maxsub{max_sub}_depth{depth}_synonyms{synonyms}_noise{noise}_{branching}.csv'
    return os.path.join(synth_base, fname)


SYNTHETIC_CONFIGS = {
    'ecosystems-shallow': {
        'path': synth_path('Energy_Ecosystems_and_Humans', depth=3, max_sub=5),
        'depth': 3,
    },
    'ecosystems-deep': {
        'path': synth_path('Energy_Ecosystems_and_Humans', depth=5, max_sub=3),
        'depth': 5,
    },
    'fisheries-shallow': {
        'path': synth_path('Offshore_energy_impacts_on_fisheries', depth=3, max_sub=5),
        'depth': 3,
    },
    'fisheries-deep': {
        'path': synth_path('Offshore_energy_impacts_on_fisheries', depth=5, max_sub=3),
        'depth': 5,
    },
}

for nick, cfg in SYNTHETIC_CONFIGS.items():
    print(f'{nick}: {cfg["path"]}')

ecosystems-shallow: /mnt/ffs24/home/chidurah/NCEAS_Unsupervised_NLP/data/synthetic/generated_data/Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub5_depth3_synonyms0_noise0.0_random.csv
ecosystems-deep: /mnt/ffs24/home/chidurah/NCEAS_Unsupervised_NLP/data/synthetic/generated_data/Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub3_depth5_synonyms0_noise0.0_random.csv
fisheries-shallow: /mnt/ffs24/home/chidurah/NCEAS_Unsupervised_NLP/data/synthetic/generated_data/Offshore_energy_impacts_on_fisheries_hierarchy_t1.0_maxsub5_depth3_synonyms0_noise0.0_random.csv
fisheries-deep: /mnt/ffs24/home/chidurah/NCEAS_Unsupervised_NLP/data/synthetic/generated_data/Offshore_energy_impacts_on_fisheries_hierarchy_t1.0_maxsub3_depth5_synonyms0_noise0.0_random.csv


In [5]:
synth_rows = []

for nick, cfg in SYNTHETIC_CONFIGS.items():
    try:
        df = pd.read_csv(cfg['path'])
        # Keep only rows that reach the full hierarchy depth
        deepest_col = f'category {cfg["depth"] - 1}'
        df = df.dropna(subset=[deepest_col]).reset_index(drop=True)
        df = df[df['topic'] != df[deepest_col]].reset_index(drop=True)

        row = {'Dataset': nick, 'N': len(df)}
        for i in range(cfg['depth']):
            col = f'category {i}'
            row[f'Level {i} labels'] = df[col].nunique() if col in df.columns else 'N/A'
        synth_rows.append(row)
        print(f'Loaded {nick}: {len(df):,} rows')
    except FileNotFoundError:
        row = {'Dataset': nick, 'N': 'FILE NOT FOUND'}
        row['path'] = cfg['path']
        synth_rows.append(row)
        print(f'File not found for {nick}: {cfg["path"]}')
    except Exception as e:
        print(f'Error loading {nick}: {e}')

synth_stats = pd.DataFrame(synth_rows).set_index('Dataset')
synth_stats

Loaded ecosystems-shallow: 549 rows
Loaded ecosystems-deep: 1,425 rows
Loaded fisheries-shallow: 669 rows
Loaded fisheries-deep: 1,856 rows


,N,Level 0 labels,Level 1 labels,Level 2 labels,Level 3 labels,Level 4 labels
Dataset,,,,,,
ecosystems-shallow,549,5,26,113,NaN,NaN
ecosystems-deep,1425,5,16,53,156.0,474.0
fisheries-shallow,669,6,30,137,NaN,NaN
fisheries-deep,1856,6,20,68,209.0,610.0
